# Domain C: Behavioral State Modulation (Running)

This notebook addresses two questions:
- **C1:** Does running differentially modulate responses across cell types?
- **C2:** Is the same VIP subtype that mediates running modulation also involved in context adaptation?

**Data:** Zarr multimodal stores. Each trial has `is_running` (bool) and `avg_running` (cm/s). We split trials into running vs. stationary, compute running modulation indices, characterize gain vs. additive modulation, and link VIP subtypes to multi-dimensional functional roles.

In [ ]:
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, wilcoxon, mannwhitneyu, spearmanr, pearsonr
from scipy.optimize import curve_fit
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

def load_mouse_zarr(mouse_id):
    """Load one mouse's data from zarr, returning an adata-like SimpleNamespace."""
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    ct = z['transcriptomics/cell_type']
    morph = z['morphology/mask_properties']
    obs_dict = {
        'unique_id': z['unique_id'][:].astype(str),
        'mouse_id': mouse_id,
        'class_name': ct['class_name'][:],
        'subclass_name': ct['subclass_name'][:],
        'subclass_label': ct['subclass_label'][:],
        'supertype_name': ct['supertype_name'][:],
        'cluster_name': ct['cluster_name'][:],
        'cluster_alias': ct['cluster_alias'][:],
        'x_loc': morph['centroid_x_um'][:],
        'y_loc': morph['centroid_y_um'][:],
        'z_loc': morph['centroid_z_um'][:],
    }
    # Gene expression
    gene_names = sorted(z['transcriptomics/cellxgene'].keys())
    for g in gene_names:
        obs_dict[g] = z[f'transcriptomics/cellxgene/{g}'][:]
    obs_df = pd.DataFrame(obs_dict)
    n_cells = len(obs_df)

    X_sessions, var_sessions = [], []
    for si, sess in enumerate(SESSIONS):
        gs = z[f'ophys/drifting_gratings/{sess}/stim_aligned_dff/GratingStim']
        dff = gs['dff'][:]
        time_rel = gs['time_relative'][:]
        running = gs['running'][:]
        gray = gs['trial_info/gray_screen'][:]
        valid = ~gray
        stim_mask = (time_rel >= 0) & (time_rel <= 2.0)
        dff_avg = dff[valid][:, stim_mask, :].mean(axis=1)
        run_speed = running[valid][:, stim_mask, 0].mean(axis=1)
        X_sessions.append(dff_avg.T)
        var_sessions.append(pd.DataFrame({
            'contrast': gs['trial_info/contrast'][:][valid],
            'orientation': gs['trial_info/orientation'][:][valid],
            'temporal_frequency': gs['trial_info/temporal_frequency'][:][valid],
            'spatial_frequency': gs['trial_info/spatial_frequency'][:][valid],
            'stim_block': gs['trial_info/stim_block'][:][valid],
            'stim_name': gs['trial_info/stim_name'][:][valid],
            'start_time': gs['trial_info/start_time'][:][valid],
            'stop_time': gs['trial_info/stop_time'][:][valid],
            'duration': gs['trial_info/duration'][:][valid],
            'avg_running': run_speed,
            'is_running': run_speed > 1.0,
            'day': si + 1,
        }))
    return SimpleNamespace(
        X=np.hstack(X_sessions), obs=obs_df, var=pd.concat(var_sessions, ignore_index=True),
        n_obs=n_cells, n_vars=sum(v.shape[0] for v in var_sessions)
    )

adata_list = [load_mouse_zarr(mid) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

# ── Subclass setup ──
SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

print(f"Total cells: {len(obs)}")
print(f"Running trials: {var['is_running'].sum()}, Stationary: {(~var['is_running']).sum()}")

## C1: Cell-Type-Specific Running Modulation

Compute running modulation index (RMI) per cell, decompose into gain vs additive components, and compare running-modulated tuning curves across subclasses.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C1.1  Running Modulation Index (RMI) per cell
# ══════════════════════════════════════════════════════════════════════

run_mask = var['is_running'].values.astype(bool)
stat_mask = ~run_mask

# Overall RMI: (mean_run - mean_stat) / (|mean_run| + |mean_stat|)
mean_run = np.nanmean(X[:, run_mask], axis=1)
mean_stat = np.nanmean(X[:, stat_mask], axis=1)
denom = np.abs(mean_run) + np.abs(mean_stat)
denom[denom < 1e-8] = np.nan
rmi = (mean_run - mean_stat) / denom

# Per-condition RMI (orientation × running state, using contrast-context blocks)
contrast_blocks = var['stim_block'].isin([0.0, 2.0])
rmi_per_ori = np.zeros((adata.n_obs, len(orientations)))
ori_run_resp = np.zeros((adata.n_obs, len(orientations)))
ori_stat_resp = np.zeros((adata.n_obs, len(orientations)))

for i, ori in enumerate(orientations):
    run_trials = np.where((contrast_blocks & (var['orientation'] == ori) & run_mask).values)[0]
    stat_trials = np.where((contrast_blocks & (var['orientation'] == ori) & stat_mask).values)[0]
    
    r_run = np.nanmean(X[:, run_trials], axis=1) if len(run_trials) > 0 else np.zeros(adata.n_obs)
    r_stat = np.nanmean(X[:, stat_trials], axis=1) if len(stat_trials) > 0 else np.zeros(adata.n_obs)
    
    ori_run_resp[:, i] = r_run
    ori_stat_resp[:, i] = r_stat
    d = np.abs(r_run) + np.abs(r_stat)
    d[d < 1e-8] = np.nan
    rmi_per_ori[:, i] = (r_run - r_stat) / d

rmi_mean = np.nanmean(rmi_per_ori, axis=1)

run_df = pd.DataFrame({
    'RMI': rmi,
    'RMI_per_ori': rmi_mean,
    'mean_run': mean_run,
    'mean_stat': mean_stat,
    'subclass': obs['subclass_name'].values,
    'subclass_short': obs['subclass_name'].map(SUBCLASS_SHORT).values,
    'mouse_id': obs['mouse_id'].values,
    'supertype': obs['supertype_name'].values,
    'cluster': obs['cluster_name'].values,
})

print("RMI statistics by subclass:")
print(run_df.groupby('subclass_short')['RMI'].describe().round(3))

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C1.2  Visualization: RMI distributions & running-state tuning curves
# ══════════════════════════════════════════════════════════════════════

# ── RMI violin plot by subclass ──
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

ax = axes[0]
sns.violinplot(data=run_df[run_df['subclass'].isin(present_subclasses)],
               x='subclass_short', y='RMI', order=short_order,
               palette=short_pal, cut=0, inner='box', ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.5, lw=1)
ax.set_title('C1: Running Modulation Index', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('RMI (+: enhanced by running)')
ax.tick_params(axis='x', rotation=45)

# ── Scatter: stationary vs running response ──
ax = axes[1]
for sc in present_subclasses:
    mask = run_df['subclass'] == sc
    ax.scatter(run_df.loc[mask, 'mean_stat'], run_df.loc[mask, 'mean_run'],
               alpha=0.3, s=15, c=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc])
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k--', alpha=0.4, lw=1)
ax.set_xlabel('Mean ΔF/F (Stationary)')
ax.set_ylabel('Mean ΔF/F (Running)')
ax.set_title('C1: Running vs Stationary Response', fontweight='bold')
ax.legend(fontsize=7, loc='upper left')

# ── RMI per cell histogram, pooled ──
ax = axes[2]
for sc in present_subclasses:
    mask = run_df['subclass'] == sc
    ax.hist(run_df.loc[mask, 'RMI'].dropna(), bins=30, alpha=0.4,
            color=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc], density=True)
ax.axvline(0, color='k', ls='--', alpha=0.5)
ax.set_xlabel('RMI')
ax.set_ylabel('Density')
ax.set_title('C1: RMI Distribution', fontweight='bold')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

# ── Kruskal-Wallis test ──
groups = [run_df.loc[run_df['subclass'] == s, 'RMI'].dropna().values for s in present_subclasses]
groups = [g for g in groups if len(g) >= 3]
stat, p = kruskal(*groups)
print(f"RMI Kruskal-Wallis across subclasses: H={stat:.2f}, p={p:.2e}")

# ── One-sample Wilcoxon (RMI ≠ 0) per subclass ──
print("\nRMI ≠ 0 (Wilcoxon):")
for sc in present_subclasses:
    vals = run_df.loc[run_df['subclass'] == sc, 'RMI'].dropna().values
    if len(vals) >= 10:
        stat, p = wilcoxon(vals)
        print(f"  {SUBCLASS_SHORT[sc]:10s}: median={np.median(vals):+.4f}, p={p:.2e}")

# ── Orientation tuning: running vs stationary per subclass ──
fig, axes = plt.subplots(2, len(present_subclasses),
                         figsize=(4*len(present_subclasses), 8), sharey='row')
for col_i, sc in enumerate(present_subclasses):
    mask = obs['subclass_name'].values == sc
    n_sc = mask.sum()
    
    # Orientation tuning
    ax = axes[0, col_i]
    mean_run_o = np.nanmean(ori_run_resp[mask], axis=0)
    sem_run_o = np.nanstd(ori_run_resp[mask], axis=0) / np.sqrt(n_sc)
    mean_stat_o = np.nanmean(ori_stat_resp[mask], axis=0)
    sem_stat_o = np.nanstd(ori_stat_resp[mask], axis=0) / np.sqrt(n_sc)
    
    ax.errorbar(orientations, mean_run_o, yerr=sem_run_o, color='red', label='Running',
                linewidth=2, capsize=3, marker='o')
    ax.errorbar(orientations, mean_stat_o, yerr=sem_stat_o, color='blue', label='Stationary',
                linewidth=2, capsize=3, marker='s')
    ax.set_title(SUBCLASS_SHORT[sc], color=SUBCLASS_COLORS[sc], fontweight='bold')
    ax.set_xlabel('Direction (°)')
    ax.set_xticks(orientations[::2])
    if col_i == 0:
        ax.set_ylabel('Mean ΔF/F')
        ax.legend(fontsize=7)
    
    # RMI per orientation
    ax = axes[1, col_i]
    mean_rmi_o = np.nanmean(rmi_per_ori[mask], axis=0)
    sem_rmi_o = np.nanstd(rmi_per_ori[mask], axis=0) / np.sqrt(n_sc)
    ax.bar(orientations, mean_rmi_o, width=30, color=SUBCLASS_COLORS[sc], alpha=0.7)
    ax.errorbar(orientations, mean_rmi_o, yerr=sem_rmi_o, fmt='none', color='k', capsize=3)
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    ax.set_xlabel('Direction (°)')
    ax.set_xticks(orientations[::2])
    if col_i == 0:
        ax.set_ylabel('RMI')

plt.suptitle('C1: Running-State Orientation Tuning & RMI per Direction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C1.3  Gain vs Additive decomposition of running modulation
#       For each cell: R(stim, run) = gain * R(stim, stat) + offset
# ══════════════════════════════════════════════════════════════════════

gain_arr = np.full(adata.n_obs, np.nan)
offset_arr = np.full(adata.n_obs, np.nan)
r2_arr = np.full(adata.n_obs, np.nan)

for i in range(adata.n_obs):
    r_stat = ori_stat_resp[i]
    r_run = ori_run_resp[i]
    valid = ~np.isnan(r_stat) & ~np.isnan(r_run)
    if valid.sum() < 4 or np.std(r_stat[valid]) < 1e-6:
        continue
    # Fit: r_run = gain * r_stat + offset
    A = np.column_stack([r_stat[valid], np.ones(valid.sum())])
    result = np.linalg.lstsq(A, r_run[valid], rcond=None)
    gain_arr[i] = result[0][0]
    offset_arr[i] = result[0][1]
    # R²
    pred = A @ result[0]
    ss_res = np.sum((r_run[valid] - pred)**2)
    ss_tot = np.sum((r_run[valid] - np.mean(r_run[valid]))**2)
    r2_arr[i] = 1 - ss_res / ss_tot if ss_tot > 0 else 0

run_df['gain'] = gain_arr
run_df['offset'] = offset_arr
run_df['gain_r2'] = r2_arr

# ── Visualization ──
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Gain by subclass
ax = axes[0]
data_plot = run_df[run_df['subclass'].isin(present_subclasses)].copy()
sns.violinplot(data=data_plot, x='subclass_short', y='gain', order=short_order,
               palette=short_pal, cut=0, inner='box', ax=ax)
ax.axhline(1, color='k', ls='--', alpha=0.5, lw=1, label='no change')
ax.set_title('C1: Multiplicative Gain', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Gain (slope)')
ax.tick_params(axis='x', rotation=45)

# Offset by subclass
ax = axes[1]
sns.violinplot(data=data_plot, x='subclass_short', y='offset', order=short_order,
               palette=short_pal, cut=0, inner='box', ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.5, lw=1)
ax.set_title('C1: Additive Offset', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Offset (intercept)')
ax.tick_params(axis='x', rotation=45)

# Gain vs Offset scatter
ax = axes[2]
for sc in present_subclasses:
    mask = run_df['subclass'] == sc
    ax.scatter(run_df.loc[mask, 'gain'], run_df.loc[mask, 'offset'],
               alpha=0.3, s=15, c=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc])
ax.axhline(0, color='k', ls='--', alpha=0.3)
ax.axvline(1, color='k', ls='--', alpha=0.3)
ax.set_xlabel('Gain')
ax.set_ylabel('Offset')
ax.set_title('C1: Gain vs Offset', fontweight='bold')
ax.legend(fontsize=7)

plt.suptitle('C1: Decomposition of Running Modulation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Statistics: is running modulation primarily gain or offset? ──
print("=== Gain (> 1 = enhanced) ===")
for sc in present_subclasses:
    vals = run_df.loc[run_df['subclass'] == sc, 'gain'].dropna()
    if len(vals) >= 10:
        stat, p = wilcoxon(vals - 1)  # test against gain=1
        print(f"  {SUBCLASS_SHORT[sc]:10s}: median gain={vals.median():.3f}, differs from 1: p={p:.2e}")

print("\n=== Offset (≠ 0) ===")
for sc in present_subclasses:
    vals = run_df.loc[run_df['subclass'] == sc, 'offset'].dropna()
    if len(vals) >= 10:
        stat, p = wilcoxon(vals)
        print(f"  {SUBCLASS_SHORT[sc]:10s}: median offset={vals.median():.4f}, p={p:.2e}")

## C2: VIP Subtype Integration Hypothesis

Test whether a specific VIP transcriptomic subtype simultaneously mediates running modulation, context adaptation, and cross-day experience dependence. Cluster VIP cells by multidimensional functional measures, map to transcriptomic subtypes, and identify distinguishing molecular signatures.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C2.1  Compute multi-dimensional functional measures for VIP cells
#       Requires adaptation index (from Domain B) and cross-day drift
# ══════════════════════════════════════════════════════════════════════

vip_mask = obs['subclass_name'].values == '046 Vip Gaba'
n_vip = vip_mask.sum()
print(f"VIP cells: {n_vip}")
print(f"VIP supertypes: {obs.loc[vip_mask, 'supertype_name'].value_counts().to_dict()}")
print(f"VIP clusters: {obs.loc[vip_mask, 'cluster_name'].value_counts().to_dict()}")

# ── RMI for VIP cells (already computed above) ──
vip_rmi = run_df.loc[vip_mask, 'RMI'].values

# ── Adaptation index for VIP cells ──
# Recompute block-level responses for adaptation (same as Domain B)
def get_block_responses_simple(X, var, block, param_name, param_values, orientations):
    block_mask = var['stim_block'] == block
    responses = {}
    for ori in orientations:
        for pval in param_values:
            mask = block_mask & (var['orientation'] == ori) & (var[param_name] == pval)
            trial_idx = np.where(mask.values)[0]
            if len(trial_idx) > 0:
                responses[(ori, pval)] = np.nanmean(X[:, trial_idx], axis=1)
            else:
                responses[(ori, pval)] = np.full(X.shape[0], np.nan)
    return responses

resp_b0 = get_block_responses_simple(X, var, 0.0, 'contrast', contrasts, orientations)
resp_b2 = get_block_responses_simple(X, var, 2.0, 'contrast', contrasts, orientations)
conditions = [(ori, c) for ori in orientations for c in contrasts]

adapt_all = []
for cond in conditions:
    r1, r2 = resp_b0[cond], resp_b2[cond]
    d = np.abs(r1) + np.abs(r2)
    d[d < 1e-8] = np.nan
    adapt_all.append((r1 - r2) / d)
adapt_matrix = np.column_stack(adapt_all)
adapt_idx = np.nanmean(adapt_matrix, axis=1)
vip_adapt = adapt_idx[vip_mask]

# ── Cross-day drift for VIP cells ──
days = sorted(var['day'].unique())
day_ori_responses = {}
for day in days:
    day_mask = var['day'] == day
    contrast_day = day_mask & var['stim_block'].isin([0.0, 2.0])
    resp = np.zeros((adata.n_obs, len(orientations)))
    for i, ori in enumerate(orientations):
        mask_d = contrast_day & (var['orientation'] == ori)
        trial_idx = np.where(mask_d.values)[0]
        if len(trial_idx) > 0:
            resp[:, i] = np.nanmean(X[:, trial_idx], axis=1)
    day_ori_responses[day] = resp

# Drift = mean pairwise correlation depreciation
from itertools import combinations
vip_drift = np.full(n_vip, np.nan)
vip_indices = np.where(vip_mask)[0]
for vi, cell_i in enumerate(vip_indices):
    corrs = []
    for d1, d2 in combinations(days, 2):
        v1 = day_ori_responses[d1][cell_i]
        v2 = day_ori_responses[d2][cell_i]
        if np.std(v1) > 1e-6 and np.std(v2) > 1e-6:
            r, _ = pearsonr(v1, v2)
            corrs.append(r)
    if corrs:
        vip_drift[vi] = 1 - np.mean(corrs)  # Higher = more drift

# ── Assemble VIP functional profile ──
vip_func_df = pd.DataFrame({
    'RMI': vip_rmi,
    'adaptation': vip_adapt,
    'drift': vip_drift,
    'supertype': obs.loc[vip_mask, 'supertype_name'].values,
    'cluster': obs.loc[vip_mask, 'cluster_name'].values,
    'mouse_id': obs.loc[vip_mask, 'mouse_id'].values,
})
print("\nVIP functional measures:")
print(vip_func_df.describe())

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C2.2  Cluster VIP cells by functional measures & map to transcriptomics
# ══════════════════════════════════════════════════════════════════════

# Prepare feature matrix for clustering
func_features = vip_func_df[['RMI', 'adaptation', 'drift']].copy()
valid_vip = func_features.notna().all(axis=1)
func_valid = func_features[valid_vip].values
n_valid = valid_vip.sum()
print(f"VIP cells with all 3 measures: {n_valid}")

if n_valid >= 6:
    scaler = StandardScaler()
    func_scaled = scaler.fit_transform(func_valid)
    
    # Hierarchical clustering (k=2 or 3 depending on sample size)
    n_clusters = min(3, max(2, n_valid // 5))
    hc = AgglomerativeClustering(n_clusters=n_clusters)
    cluster_labels = hc.fit_predict(func_scaled)
    vip_func_df.loc[valid_vip, 'func_cluster'] = cluster_labels
    
    # ── 3D scatter: RMI × adaptation × drift ──
    from mpl_toolkits.mplot3d import Axes3D
    fig = plt.figure(figsize=(16, 6))
    
    # Color by functional cluster
    ax1 = fig.add_subplot(131, projection='3d')
    for cl in range(n_clusters):
        mask = cluster_labels == cl
        ax1.scatter(func_valid[mask, 0], func_valid[mask, 1], func_valid[mask, 2],
                   alpha=0.7, s=50, label=f'Cluster {cl}')
    ax1.set_xlabel('RMI')
    ax1.set_ylabel('Adaptation')
    ax1.set_zlabel('Drift')
    ax1.set_title('Functional Clusters')
    ax1.legend(fontsize=8)
    
    # Color by supertype
    ax2 = fig.add_subplot(132, projection='3d')
    supertypes_valid = vip_func_df.loc[valid_vip, 'supertype'].values
    unique_st = np.unique(supertypes_valid)
    st_colors = plt.cm.tab10(np.linspace(0, 1, len(unique_st)))
    for si, st in enumerate(unique_st):
        mask = supertypes_valid == st
        ax2.scatter(func_valid[mask, 0], func_valid[mask, 1], func_valid[mask, 2],
                   alpha=0.7, s=50, color=st_colors[si], label=st[:20])
    ax2.set_xlabel('RMI')
    ax2.set_ylabel('Adaptation')
    ax2.set_zlabel('Drift')
    ax2.set_title('By Supertype')
    ax2.legend(fontsize=6)
    
    # 2D pairwise
    ax3 = fig.add_subplot(133)
    for cl in range(n_clusters):
        mask = cluster_labels == cl
        ax3.scatter(func_valid[mask, 0], func_valid[mask, 1], alpha=0.7, s=50, label=f'Cluster {cl}')
    ax3.set_xlabel('RMI')
    ax3.set_ylabel('Adaptation Index')
    ax3.set_title('RMI vs Adaptation')
    ax3.legend(fontsize=8)
    
    plt.suptitle('C2: VIP Multidimensional Functional Clustering', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # ── Cross-tabulate functional clusters vs transcriptomic types ──
    valid_df = vip_func_df[valid_vip].copy()
    print("\n=== Functional Cluster × Supertype ===")
    ct = pd.crosstab(valid_df['func_cluster'], valid_df['supertype'])
    print(ct)
    
    print("\n=== Functional Cluster × Cluster ===")
    ct2 = pd.crosstab(valid_df['func_cluster'], valid_df['cluster'])
    print(ct2)
    
    # ── Functional measure comparison across clusters ──
    print("\n=== Functional measures per cluster ===")
    for cl in range(n_clusters):
        cl_data = valid_df[valid_df['func_cluster'] == cl]
        print(f"  Cluster {cl} (n={len(cl_data)}): RMI={cl_data['RMI'].mean():.3f}±{cl_data['RMI'].std():.3f}, "
              f"adapt={cl_data['adaptation'].mean():.3f}±{cl_data['adaptation'].std():.3f}, "
              f"drift={cl_data['drift'].mean():.3f}±{cl_data['drift'].std():.3f}")
    
    # ── Correlation among modulations within VIP ──
    print("\n=== Correlations among VIP functional measures ===")
    for m1, m2 in [('RMI', 'adaptation'), ('RMI', 'drift'), ('adaptation', 'drift')]:
        r, p = spearmanr(func_valid[:, ['RMI','adaptation','drift'].index(m1)],
                        func_valid[:, ['RMI','adaptation','drift'].index(m2)])
        print(f"  {m1} vs {m2}: ρ={r:.3f}, p={p:.3e}")
else:
    print("Not enough VIP cells with complete data for clustering.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C2.3  Gene expression signatures of VIP functional subtypes
#       & Spatial proximity analysis (VIP → Sst disinhibition circuit)
# ══════════════════════════════════════════════════════════════════════

META_COLS = ['unique_id', 'mouse_id', 'class_name',
             'subclass_name', 'subclass_label',
             'supertype_name', 'cluster_name', 'cluster_alias',
             'x_loc', 'y_loc', 'z_loc']
GENE_COLS = [c for c in obs.columns if c not in META_COLS
             and not c.startswith('NegControl') and not c.startswith('UnassignedCodeword')]

if n_valid >= 6 and 'func_cluster' in vip_func_df.columns:
    # Differential gene expression between VIP functional clusters
    vip_obs = obs[vip_mask].copy()
    vip_genes = vip_obs[GENE_COLS].values.astype(float)
    valid_indices = np.where(valid_vip.values)[0]
    
    # Compare cluster 0 vs others
    unique_clusters = sorted(vip_func_df.loc[valid_vip, 'func_cluster'].dropna().unique())
    if len(unique_clusters) >= 2:
        diff_results = []
        for gi, gene in enumerate(GENE_COLS):
            gx = vip_genes[:, gi]
            if np.std(gx) < 1e-6:
                continue
            for ci, cl in enumerate(unique_clusters):
                cl_mask = vip_func_df['func_cluster'].values == cl
                other_mask = vip_func_df['func_cluster'].notna().values & ~cl_mask
                if cl_mask.sum() >= 3 and other_mask.sum() >= 3:
                    v1 = gx[cl_mask[:len(gx)]] if cl_mask.sum() <= len(gx) else gx
                    v2 = gx[other_mask[:len(gx)]] if other_mask.sum() <= len(gx) else gx
                    try:
                        stat, p = mannwhitneyu(v1, v2, alternative='two-sided')
                        diff_results.append({'gene': gene, 'cluster': cl,
                                           'mean_cl': np.mean(v1), 'mean_other': np.mean(v2),
                                           'logFC': np.log2((np.mean(v1)+0.1)/(np.mean(v2)+0.1)),
                                           'p': p})
                    except:
                        pass
        
        if diff_results:
            diff_df = pd.DataFrame(diff_results)
            diff_df['p_adj'] = diff_df.groupby('cluster')['p'].transform(
                lambda x: np.minimum(x * len(x), 1))  # Bonferroni
            top_genes = diff_df.sort_values('p').head(20)
            print("Top differentially expressed genes between VIP functional clusters:")
            print(top_genes[['gene', 'cluster', 'logFC', 'p', 'p_adj']].to_string())
            
            # Heatmap of top genes
            top_gene_names = top_genes['gene'].unique()[:15]
            heatmap_data = pd.DataFrame(index=top_gene_names)
            for cl in unique_clusters:
                cl_cells = valid_indices[cluster_labels == cl]
                for g in top_gene_names:
                    gi = GENE_COLS.index(g)
                    heatmap_data.loc[g, f'Cluster {int(cl)}'] = np.mean(vip_genes[cl_cells, gi])
            
            fig, ax = plt.subplots(figsize=(8, max(5, len(top_gene_names)*0.4)))
            sns.heatmap(heatmap_data.astype(float), cmap='YlOrRd', annot=True, fmt='.2f', ax=ax)
            ax.set_title('C2: Gene Expression by VIP Functional Cluster', fontweight='bold')
            plt.tight_layout()
            plt.show()

# ── Spatial proximity: VIP ↔ Sst circuit analysis ──
vip_coords = obs.loc[vip_mask, ['x_loc', 'y_loc', 'z_loc']].values.astype(float)
sst_mask = obs['subclass_name'].values == '053 Sst Gaba'
sst_coords = obs.loc[sst_mask, ['x_loc', 'y_loc', 'z_loc']].values.astype(float)

# For each VIP cell, find distance to nearest Sst cell
from scipy.spatial.distance import cdist
if sst_mask.sum() > 0:
    dist_vip_sst = cdist(vip_coords, sst_coords)
    nearest_sst_dist = np.min(dist_vip_sst, axis=1)
    
    run_df_vip = run_df[vip_mask].copy()
    run_df_vip['nearest_sst_dist'] = nearest_sst_dist
    
    # Correlate VIP RMI with distance to nearest Sst
    valid_d = ~np.isnan(nearest_sst_dist) & ~np.isnan(run_df_vip['RMI'].values)
    if valid_d.sum() > 5:
        r, p = spearmanr(nearest_sst_dist[valid_d], run_df_vip.loc[valid_d.values, 'RMI'].values)
        print(f"\nVIP RMI vs distance to nearest Sst: ρ={r:.3f}, p={p:.3e}")
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(nearest_sst_dist, run_df_vip['RMI'].values, alpha=0.6, s=40, c='#e377c2')
    ax.set_xlabel('Distance to nearest Sst cell (µm)')
    ax.set_ylabel('VIP cell RMI')
    ax.set_title('C2: VIP Running Modulation vs Proximity to Sst', fontweight='bold')
    plt.tight_layout()
    plt.show()

## C3: Moment-to-Moment Running Speed × ΔF/F Coupling (10 Hz)

Using the 10 Hz trial-resolved running speed and ΔF/F, we examine:
- **C3.1** Within-trial running–ΔF/F cross-correlation at fine temporal lags
- **C3.2** Does ΔF/F follow instantaneous running speed changes or lag behind? How does this differ by subclass?

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C3.1  Load 10 Hz data: within-trial running–ΔF/F cross-correlation
# ══════════════════════════════════════════════════════════════════════

def load_zarr_10hz(mouse_id, session='session_1'):
    """Load 10 Hz trial-resolved data from zarr."""
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    gs = z[f'ophys/drifting_gratings/{session}/stim_aligned_dff/GratingStim']
    ti_dict = {k: gs[f'trial_info/{k}'][:] for k in gs['trial_info'].keys()}
    return {
        'dff': gs['dff'][:],
        'unique_ids': z['unique_id'][:].astype(str),
        'running': gs['running'][:],
        'time_rel': gs['time_relative'][:],
        'trial_info': pd.DataFrame(ti_dict),
    }

# Load one mouse
mouse_pick = obs['mouse_id'].value_counts().idxmax()
pk = load_zarr_10hz(mouse_pick)
dff_10hz = pk['dff']            # (n_trials, 41, n_cells)
run_10hz = pk['running'][:, :, 0]  # (n_trials, 41) — speed only
time_rel = pk['time_rel']
ti_10hz = pk['trial_info']
uids_10hz = pk['unique_ids']

# Match cells
obs_mouse = obs[obs['mouse_id'] == mouse_pick]
matched = {}
for ci, uid in enumerate(uids_10hz):
    m = obs_mouse[obs_mouse['unique_id'] == uid]
    if len(m) > 0:
        matched[ci] = m.iloc[0]['subclass_name']

matched_cells = sorted(matched.keys())
cell_sc = np.array([matched[ci] for ci in matched_cells])

# Select stimulus-period timepoints
stim_mask = (time_rel >= 0) & (time_rel <= 2.0)
stim_tp = np.where(stim_mask)[0]
n_stim_tp = len(stim_tp)

# Non-gray trials
non_gray = ~ti_10hz['gray_screen'].astype(bool)
grating_idx = np.where(non_gray.values)[0]

print(f"Mouse {mouse_pick}: {len(matched_cells)} cells, {len(grating_idx)} grating trials")
print(f"Stimulus period: {n_stim_tp} timepoints at 10 Hz")

# ── Compute cross-correlation between running speed and ΔF/F ──
max_lag = 5  # ±5 samples = ±500 ms
lags = np.arange(-max_lag, max_lag + 1) * 0.1  # in seconds

def xcorr_lagged(run_trials, dff_trials, max_lag):
    """Compute mean cross-correlation at each lag across trials."""
    n_lags = 2 * max_lag + 1
    cc = np.zeros(n_lags)
    for li, lag in enumerate(range(-max_lag, max_lag + 1)):
        rvals = []
        for tr in range(run_trials.shape[0]):
            r = run_trials[tr]
            d = dff_trials[tr]
            if lag >= 0:
                r_seg = r[:len(r)-lag] if lag > 0 else r
                d_seg = d[lag:]
            else:
                r_seg = r[-lag:]
                d_seg = d[:len(d)+lag]
            if len(r_seg) < 3 or np.std(r_seg) < 1e-6 or np.std(d_seg) < 1e-6:
                continue
            rvals.append(np.corrcoef(r_seg, d_seg)[0, 1])
        cc[li] = np.nanmean(rvals) if rvals else np.nan
    return cc

# Running during stimulus period for grating trials
run_stim = run_10hz[grating_idx][:, stim_tp]  # (n_grating, n_stim_tp)

# Compute per-cell cross-correlation
xcorr_all = np.zeros((len(matched_cells), len(lags)))
for ci_idx, ci in enumerate(matched_cells):
    dff_stim = dff_10hz[grating_idx][:, stim_tp, ci]  # (n_grating, n_stim_tp)
    xcorr_all[ci_idx] = xcorr_lagged(run_stim, dff_stim, max_lag)
    
print(f"Cross-correlations computed for {len(matched_cells)} cells")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C3.2  ΔF/F tracks instantaneous running: time-resolved gain analysis
# ══════════════════════════════════════════════════════════════════════

# ── Bin trials by running speed at each 100ms time bin and compute ΔF/F ──
# This shows whether neural modulation follows running in real time

n_speed_bins = 5
speed_bin_edges = np.percentile(run_stim.ravel(), np.linspace(0, 100, n_speed_bins + 1))
speed_bin_centers = (speed_bin_edges[:-1] + speed_bin_edges[1:]) / 2

# For each time point in the stimulus period, bin trials by running speed
# and compute mean ΔF/F for each subclass
fig, axes = plt.subplots(1, len(present_sc_c), figsize=(5 * len(present_sc_c), 5), sharey=True)
if len(present_sc_c) == 1:
    axes = [axes]

for ax, sc in zip(axes, present_sc_c):
    sc_mask = cell_sc == sc
    sc_cells_idx = np.array(matched_cells)[sc_mask]
    if len(sc_cells_idx) < 3:
        ax.set_visible(False)
        continue
    
    # For each time bin, compute speed–ΔF/F curve
    time_bins = [0, 5, 10, 15]  # indices into stim_tp, roughly 0, 0.5, 1.0, 1.5 s
    time_labels = ['0–0.5s', '0.5–1.0s', '1.0–1.5s', '1.5–2.0s']
    cmap = plt.cm.viridis(np.linspace(0.2, 0.9, len(time_bins)))
    
    for ti, (t_start, t_label, color) in enumerate(zip(time_bins, time_labels, cmap)):
        t_end = t_start + 5
        if t_end > n_stim_tp:
            continue
        tp_range = stim_tp[t_start:t_end]
        run_bin = np.nanmean(run_stim[:, t_start:t_end], axis=1)  # (n_trials,)
        dff_bin = np.nanmean(
            dff_10hz[grating_idx][:, tp_range][:, :, sc_cells_idx].mean(axis=2),
            axis=1)  # (n_trials,)
        
        # Bin by running speed
        mean_dff_per_bin = []
        for bi in range(n_speed_bins):
            bm = (run_bin >= speed_bin_edges[bi]) & (run_bin < speed_bin_edges[bi + 1])
            if bm.sum() > 5:
                mean_dff_per_bin.append(np.nanmean(dff_bin[bm]))
            else:
                mean_dff_per_bin.append(np.nan)
        
        ax.plot(speed_bin_centers, mean_dff_per_bin, 'o-', color=color, linewidth=2,
                label=t_label, markersize=5)
    
    ax.set_xlabel('Running Speed (cm/s)')
    ax.set_title(SUBCLASS_SHORT[sc], color=SUBCLASS_COLORS[sc], fontweight='bold')
    ax.legend(fontsize=7, title='Time Window')

axes[0].set_ylabel('Mean ΔF/F')
plt.suptitle('C3: Time-Resolved Running–ΔF/F Relationship', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Running onset-triggered average ──
# Find trials where running speed crosses a threshold during the stimulus
# and align ΔF/F to that onset
run_threshold = 5.0  # cm/s
onset_psths = {sc: [] for sc in present_sc_c}

for tr in range(len(grating_idx)):
    trial = grating_idx[tr]
    r = run_stim[tr]  # (n_stim_tp,)
    # Find first crossing from below to above threshold
    below = r < run_threshold
    above = r >= run_threshold
    crossings = np.where(below[:-1] & above[1:])[0]
    if len(crossings) == 0 or crossings[0] < 3 or crossings[0] > n_stim_tp - 5:
        continue
    onset_t = crossings[0]
    
    for sci, sc in enumerate(present_sc_c):
        sc_mask = cell_sc == sc
        sc_cells_idx = np.array(matched_cells)[sc_mask]
        if len(sc_cells_idx) < 3:
            continue
        # ΔF/F aligned to running onset (±3 samples)
        start = stim_tp[max(0, onset_t - 3)]
        end = stim_tp[min(n_stim_tp - 1, onset_t + 4)]
        window = np.arange(start, end + 1)
        if len(window) == 8:
            snippet = np.nanmean(dff_10hz[trial, window][:, sc_cells_idx], axis=1)
            onset_psths[sc].append(snippet)

fig, ax = plt.subplots(figsize=(10, 5))
onset_lags = np.arange(-3, 5) * 0.1  # seconds
for sc in present_sc_c:
    snippets = np.array(onset_psths[sc])
    if len(snippets) < 10:
        continue
    mean_s = np.nanmean(snippets, axis=0)
    sem_s = np.nanstd(snippets, axis=0) / np.sqrt(len(snippets))
    ax.fill_between(onset_lags, mean_s - sem_s, mean_s + sem_s,
                    alpha=0.2, color=SUBCLASS_COLORS[sc])
    ax.plot(onset_lags, mean_s, color=SUBCLASS_COLORS[sc], linewidth=2,
            label=f'{SUBCLASS_SHORT[sc]} (n={len(snippets)} events)')

ax.axvline(0, color='k', ls='--', alpha=0.5, label='Running onset')
ax.set_xlabel('Time from running onset (s)')
ax.set_ylabel('Mean ΔF/F')
ax.set_title('C3: Running-Onset-Triggered ΔF/F Average', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## C4: Transcriptomic PCs and Running-State Modulation

Do the first principal components of gene expression (tPC1–5, computed separately for Glut and GABA classes from zarr gene expression data) predict running modulation? We compute tPCs via PCA on the zarr `transcriptomics/cellxgene/` gene expression, merge with per-cell RMI, and test:
- Correlation of each tPC with RMI (whole-population and within-subclass)
- Multivariate regression: do tPCs jointly predict RMI beyond cell-type identity?
- Visualization of tPC gradients colored by running modulation

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C4.1  Compute tPCs from zarr gene expression & merge with RMI
# ══════════════════════════════════════════════════════════════════════
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from sklearn.decomposition import PCA

# ── Load gene expression from zarr for all mice ──
gene_matrices = []
for mid in MOUSE_IDS:
    z = zarr.open(f'{ZARR_DIR}/{mid}_multimodal_data.zarr', 'r')
    gene_names = sorted(z['transcriptomics/cellxgene'].keys())
    gene_vals = np.column_stack([z[f'transcriptomics/cellxgene/{g}'][:] for g in gene_names])
    gene_matrices.append(gene_vals)

gene_expr = np.vstack(gene_matrices)  # (total_cells, n_genes)

# ── Compute tPCs separately for Glut and GABA classes ──
is_glut = obs['class_name'].values == 'Glutamatergic'
is_gaba = obs['class_name'].values == 'GABAergic'

n_tpc = 5
TPC_COLS_GLUT = [f'Glut_tPC{i}' for i in range(1, n_tpc + 1)]
TPC_COLS_GABA = [f'GABA_tPC{i}' for i in range(1, n_tpc + 1)]
TPC_COLS_ALL = TPC_COLS_GLUT + TPC_COLS_GABA

# Initialize with NaN
for col in TPC_COLS_ALL:
    run_df[col] = np.nan

# Glut tPCs
if is_glut.sum() > n_tpc:
    scaler_g = StandardScaler()
    gene_glut = scaler_g.fit_transform(gene_expr[is_glut])
    pca_g = PCA(n_components=n_tpc)
    tpc_glut = pca_g.fit_transform(gene_glut)
    for i in range(n_tpc):
        run_df.loc[is_glut, f'Glut_tPC{i+1}'] = tpc_glut[:, i]
    print(f"Glut tPC explained variance: {pca_g.explained_variance_ratio_}")

# GABA tPCs
if is_gaba.sum() > n_tpc:
    scaler_b = StandardScaler()
    gene_gaba = scaler_b.fit_transform(gene_expr[is_gaba])
    pca_b = PCA(n_components=n_tpc)
    tpc_gaba = pca_b.fit_transform(gene_gaba)
    for i in range(n_tpc):
        run_df.loc[is_gaba, f'GABA_tPC{i+1}'] = tpc_gaba[:, i]
    print(f"GABA tPC explained variance: {pca_b.explained_variance_ratio_}")

# ── Create unified tPC1–5 (Glut values for Glut cells, GABA for GABA) ──
for i in range(1, n_tpc + 1):
    run_df[f'tPC{i}'] = np.where(
        run_df[f'Glut_tPC{i}'].notna(),
        run_df[f'Glut_tPC{i}'],
        run_df[f'GABA_tPC{i}']
    )

# Also add class info
run_df['class_name'] = obs['class_name'].values

print(f"\ntPC columns computed from zarr gene expression. Non-NaN counts:")
for c in [f'tPC{i}' for i in range(1, 6)]:
    print(f"  {c}: {run_df[c].notna().sum()} / {len(run_df)}")
print(f"\nGlut cells: {is_glut.sum()}")
print(f"GABA cells: {is_gaba.sum()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C4.2  Correlation of each tPC with RMI (whole-population & within-subclass)
# ══════════════════════════════════════════════════════════════════════

tpc_names = [f'tPC{i}' for i in range(1, 6)]

# ── Whole-population Spearman correlations ──
print("=== Whole-population: tPC vs RMI (Spearman) ===")
whole_corr = []
for tpc in tpc_names:
    valid = run_df[tpc].notna() & run_df['RMI'].notna()
    if valid.sum() < 20:
        continue
    rho, p = spearmanr(run_df.loc[valid, tpc], run_df.loc[valid, 'RMI'])
    whole_corr.append({'tPC': tpc, 'rho': rho, 'p': p, 'n': valid.sum()})
    print(f"  {tpc}: ρ = {rho:.3f}, p = {p:.2e}, n = {valid.sum()}")

# ── Separate by class (Glut vs GABA) ──
print("\n=== By class: tPC vs RMI ===")
class_corr = []
for cls_label, tpc_prefix in [('Glut', 'Glut_tPC'), ('GABA', 'GABA_tPC')]:
    for i in range(1, 6):
        col = f'{tpc_prefix}{i}'
        valid = run_df[col].notna() & run_df['RMI'].notna()
        if valid.sum() < 10:
            continue
        rho, p = spearmanr(run_df.loc[valid, col], run_df.loc[valid, 'RMI'])
        class_corr.append({'class': cls_label, 'tPC': f'tPC{i}', 'rho': rho, 'p': p, 'n': valid.sum()})
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f"  {cls_label} {f'tPC{i}'}: ρ = {rho:.3f}, p = {p:.2e} {sig}, n = {valid.sum()}")

class_corr_df = pd.DataFrame(class_corr)

# ── Within-subclass correlations (control for cell-type identity) ──
print("\n=== Within-subclass: unified tPC vs RMI ===")
within_corr = []
for sc in present_subclasses:
    sc_mask = run_df['subclass'] == sc
    for tpc in tpc_names:
        valid = sc_mask & run_df[tpc].notna() & run_df['RMI'].notna()
        if valid.sum() < 10:
            continue
        rho, p = spearmanr(run_df.loc[valid, tpc], run_df.loc[valid, 'RMI'])
        within_corr.append({'subclass': SUBCLASS_SHORT[sc], 'tPC': tpc, 'rho': rho, 'p': p, 'n': valid.sum()})

within_corr_df = pd.DataFrame(within_corr)
if len(within_corr_df) > 0:
    # FDR correction
    _, pvals_adj, _, _ = multipletests(within_corr_df['p'], method='fdr_bh')
    within_corr_df['p_adj'] = pvals_adj
    within_corr_df['sig'] = within_corr_df['p_adj'] < 0.05
    print(within_corr_df.to_string(index=False))

# ══════════════════════════════════════════════════════════════════════
# Visualization: Heatmap of tPC–RMI correlations by subclass
# ══════════════════════════════════════════════════════════════════════
if len(within_corr_df) > 0:
    pivot = within_corr_df.pivot(index='subclass', columns='tPC', values='rho')
    pivot_p = within_corr_df.pivot(index='subclass', columns='tPC', values='p_adj')
    
    # Annotate with stars
    annot = pivot.copy().round(2).astype(str)
    for idx in pivot.index:
        for col in pivot.columns:
            if idx in pivot_p.index and col in pivot_p.columns:
                p_val = pivot_p.loc[idx, col]
                if pd.notna(p_val) and p_val < 0.05:
                    annot.loc[idx, col] += ' *'
    
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(pivot.astype(float), annot=annot, fmt='', cmap='RdBu_r', center=0,
                vmin=-0.4, vmax=0.4, ax=ax, linewidths=0.5)
    ax.set_title('C4: tPC–RMI Spearman Correlations (within subclass)\n* = FDR < 0.05',
                 fontweight='bold')
    ax.set_ylabel('Subclass')
    ax.set_xlabel('Transcriptomic PC')
    plt.tight_layout()
    plt.show()

# ── Class-separated heatmap ──
if len(class_corr_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, cls_label in zip(axes, ['Glut', 'GABA']):
        sub = class_corr_df[class_corr_df['class'] == cls_label]
        if len(sub) == 0:
            ax.set_visible(False)
            continue
        ax.bar(sub['tPC'], sub['rho'],
               color=['#d62728' if p < 0.05 else '#aaaaaa' for p in sub['p']],
               edgecolor='black')
        ax.axhline(0, color='k', linewidth=0.5)
        ax.set_ylabel('Spearman ρ with RMI')
        ax.set_title(f'{cls_label} cells', fontweight='bold')
        ax.set_ylim(-0.3, 0.3)
        for _, row in sub.iterrows():
            if row['p'] < 0.05:
                ax.text(row['tPC'], row['rho'] + 0.01 * np.sign(row['rho']),
                        f"p={row['p']:.1e}", ha='center', fontsize=8)
    plt.suptitle('C4: tPC–RMI Correlations by Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C4.3  Multivariate regression: do tPCs predict RMI beyond subclass?
# ══════════════════════════════════════════════════════════════════════

# ── OLS: RMI ~ tPC1 + ... + tPC5 + subclass (dummy-coded) ──
# Run separately for Glut and GABA classes

for cls_label, tpc_prefix in [('Glut', 'Glut_tPC'), ('GABA', 'GABA_tPC')]:
    tpc_cols = [f'{tpc_prefix}{i}' for i in range(1, 6)]
    cls_mask = run_df[tpc_cols[0]].notna() & run_df['RMI'].notna()
    df_cls = run_df.loc[cls_mask].copy()
    
    if len(df_cls) < 30:
        print(f"\n{cls_label}: too few cells ({len(df_cls)}), skipping regression")
        continue
    
    # Model 1: subclass only
    dummies = pd.get_dummies(df_cls['subclass_short'], drop_first=True).astype(float)
    X_sub = sm.add_constant(dummies)
    model_sub = sm.OLS(df_cls['RMI'].values, X_sub).fit()
    
    # Model 2: subclass + tPCs
    X_full = sm.add_constant(pd.concat([dummies, df_cls[tpc_cols].astype(float)], axis=1))
    model_full = sm.OLS(df_cls['RMI'].values, X_full).fit()
    
    # F-test for additional variance from tPCs
    from scipy.stats import f as f_dist
    r2_sub = model_sub.rsquared
    r2_full = model_full.rsquared
    n = len(df_cls)
    p_sub = X_sub.shape[1]
    p_full = X_full.shape[1]
    df1 = p_full - p_sub
    df2 = n - p_full
    if df2 > 0 and r2_sub < 1.0:
        F_stat = ((r2_full - r2_sub) / df1) / ((1 - r2_full) / df2)
        p_value = 1 - f_dist.cdf(F_stat, df1, df2)
    else:
        F_stat, p_value = np.nan, np.nan
    
    print(f"\n{'='*60}")
    print(f"{cls_label} cells (n={len(df_cls)})")
    print(f"{'='*60}")
    print(f"  Subclass-only R²:      {r2_sub:.4f}")
    print(f"  Subclass + tPCs R²:    {r2_full:.4f}")
    print(f"  ΔR² from tPCs:         {r2_full - r2_sub:.4f}")
    print(f"  F-test: F({df1},{df2}) = {F_stat:.2f}, p = {p_value:.2e}")
    print(f"\n  tPC coefficients in full model:")
    for tpc_col in tpc_cols:
        if tpc_col in model_full.params.index:
            coef = model_full.params[tpc_col]
            pv = model_full.pvalues[tpc_col]
            sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else 'ns'
            print(f"    {tpc_col}: β = {coef:.4f}, p = {pv:.2e} {sig}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C4.4  Visualization: tPC gradients colored by running modulation
# ══════════════════════════════════════════════════════════════════════

# ── Scatter: tPC1 vs tPC2of, colored by RMI ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, cls_label, tpc_prefix in zip(axes, ['Glut', 'GABA'],
                                      ['Glut_tPC', 'GABA_tPC']):
    valid = (run_df[f'{tpc_prefix}1'].notna() &
             run_df[f'{tpc_prefix}2'].notna() &
             run_df['RMI'].notna())
    df_sub = run_df.loc[valid]
    
    if len(df_sub) < 10:
        ax.set_visible(False)
        continue
    
    sc = ax.scatter(df_sub[f'{tpc_prefix}1'], df_sub[f'{tpc_prefix}2'],
                    c=df_sub['RMI'], cmap='RdBu_r', vmin=-1, vmax=1,
                    alpha=0.6, s=15, edgecolors='none')
    plt.colorbar(sc, ax=ax, label='RMI', shrink=0.8)
    ax.set_xlabel(f'{cls_label} tPC1')
    ax.set_ylabel(f'{cls_label} tPC2')
    ax.set_title(f'{cls_label} cells (n={len(df_sub)})', fontweight='bold')

plt.suptitle('C4: Transcriptomic PC Space Colored by Running Modulation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── tPC1 vs RMI scatter per subclass ──
fig, axes = plt.subplots(1, len(present_subclasses), figsize=(4 * len(present_subclasses), 4),
                         sharey=True)
if len(present_subclasses) == 1:
    axes = [axes]

for ax, sc_name in zip(axes, present_subclasses):
    sc_mask = (run_df['subclass'] == sc_name) & run_df['tPC1'].notna() & run_df['RMI'].notna()
    df_sub = run_df.loc[sc_mask]
    if len(df_sub) < 5:
        ax.set_visible(False)
        continue
    
    ax.scatter(df_sub['tPC1'], df_sub['RMI'], alpha=0.5, s=20,
               color=SUBCLASS_COLORS[sc_name], edgecolors='none')
    
    # Add regression line
    if len(df_sub) >= 10:
        z = np.polyfit(df_sub['tPC1'], df_sub['RMI'], 1)
        x_line = np.linspace(df_sub['tPC1'].min(), df_sub['tPC1'].max(), 50)
        ax.plot(x_line, np.polyval(z, x_line), 'k--', linewidth=1.5, alpha=0.7)
        rho, p = spearmanr(df_sub['tPC1'], df_sub['RMI'])
        ax.set_title(f'{SUBCLASS_SHORT[sc_name]}\nρ={rho:.2f}, p={p:.1e}',
                     fontweight='bold', color=SUBCLASS_COLORS[sc_name], fontsize=10)
    else:
        ax.set_title(SUBCLASS_SHORT[sc_name], fontweight='bold',
                     color=SUBCLASS_COLORS[sc_name])
    
    ax.axhline(0, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel('tPC1')

axes[0].set_ylabel('RMI')
plt.suptitle('C4: tPC1 vs Running Modulation Index (per Subclass)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Summary: tPC1 distribution split by running-enhanced vs running-suppressed ──
run_df['RMI_group'] = pd.cut(run_df['RMI'], bins=[-np.inf, -0.1, 0.1, np.inf],
                              labels=['Suppressed', 'Neutral', 'Enhanced'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cls_label, tpc_col in zip(axes, ['Glut', 'GABA'], ['Glut_tPC1', 'GABA_tPC1']):
    valid = run_df[tpc_col].notna() & run_df['RMI_group'].notna()
    df_sub = run_df.loc[valid]
    if len(df_sub) < 10:
        ax.set_visible(False)
        continue
    
    sns.violinplot(data=df_sub, x='RMI_group', y=tpc_col,
                   order=['Suppressed', 'Neutral', 'Enhanced'],
                   palette={'Suppressed': '#2166ac', 'Neutral': '#aaaaaa', 'Enhanced': '#b2182b'},
                   cut=0, inner='box', ax=ax)
    ax.set_title(f'{cls_label} tPC1 by Running Modulation', fontweight='bold')
    ax.set_xlabel('Running Modulation Group')
    ax.set_ylabel(f'{cls_label} tPC1')
    
    # Stats: Kruskal-Wallis
    groups = [df_sub.loc[df_sub['RMI_group'] == g, tpc_col].dropna().values
              for g in ['Suppressed', 'Neutral', 'Enhanced']]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) >= 2:
        stat, p = kruskal(*groups)
        ax.text(0.5, 0.95, f'KW H={stat:.1f}, p={p:.2e}', transform=ax.transAxes,
                ha='center', va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('C4: tPC1 Distributions by Running Modulation Group',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()